In [ ]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from scipy.stats import linregress
import scipy.signal.signaltools

import statistics
import matplotlib.pyplot as plt
import seaborn as sns



# Define the start and end dates for the data
start_date = '2020-06-01'
end_date = '2022-05-31'

# Define the ticker symbol for the stock we want to analyze
market_prices = pd.DataFrame(data=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl"))
market_prices=market_prices[start_date:end_date]
market_returns = market_prices['adjclose'].pct_change().dropna()

ticker = 'AAPL'
stock_prices  = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/Data/PKL/Stocks/"+ticker+".pkl")
stock_prices.index = stock_prices.date
stock_prices.sort_index(inplace=True)
stock_prices=stock_prices[start_date:end_date]
stock_returns = stock_prices['adjClose'].pct_change().dropna()

# Download the Fama-French three-factor data
ff_data = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start_date, end_date)[0]

# Compute the daily excess returns for the stock and the S&P500 index
excess_stock_returns = market_returns - ff_data['RF']
excess_market_returns = market_returns - ff_data['RF']

x = pd.DataFrame({'SMB': ff_data['SMB'], 'HML': ff_data['HML']})
y = excess_stock_returns

regression = linregress(x, y)

# Compute the expected return using the Carhart four-factor model
expected_return = regression.intercept +\
                  regression.slope[1] * ff_data['SMB'].mean() + regression.slope[2] * ff_data['HML'].mean() 

In [ ]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from scipy.stats import linregress

# Define the start and end dates for the data
start_date = '2019-01-01'
end_date = '2019-12-31'

# Define the ticker symbol for the stock we want to analyze
ticker = 'AAPL'

# Download the daily price data for the stock and the S&P500 index
stock_prices = web.DataReader(ticker, 'yahoo', start_date, end_date)
market_prices = web.DataReader('^GSPC', 'yahoo', start_date, end_date)

# Compute the daily returns for the stock and the S&P500 index
stock_returns = stock_prices['Adj Close'].pct_change().dropna()
market_returns = market_prices['Adj Close'].pct_change().dropna()

# Download the Fama-French three-factor data
ff_data = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start_date, end_date)[0]

# Compute the daily excess returns for the stock and the S&P500 index
excess_stock_returns = stock_returns - ff_data['RF']
excess_market_returns = market_returns - ff_data['RF']

# Compute the daily SMB, HML, and MOM factors
ff_data['SMB'] = (ff_data['SmallCap'] - ff_data['BigCap']).div(100)
ff_data['HML'] = (ff_data['HighBM'] - ff_data['LowBM']).div(100)
ff_data['MOM'] = ff_data['Mom   '].div(100)

# Compute the regression coefficients using the Carhart four-factor model
x = pd.DataFrame({'Mkt-RF': excess_market_returns, 'SMB': ff_data['SMB'], 'HML': ff_data['HML'], 'MOM': ff_data['MOM']})
y = excess_stock_returns
regression = linregress(x, y)

# Compute the expected return using the Carhart four-factor model
expected_return = regression.intercept + regression.slope[0] * excess_market_returns.mean() + \
                  regression.slope[1] * ff_data['SMB'].mean() + regression.slope[2] * ff_data['HML'].mean() + \
                  regression.slope[3] * ff_data['MOM'].mean()

# Compute the abnormal return for the stock
abnormal_return = stock_returns.mean() - (ff_data['RF'].mean() + expected_return)

print('Abnormal return for ' + ticker + ': {:.2%}'.format(abnormal_return))


In [ ]:
ff_data

In [ ]:
stock_returns-ff_data['RF']

In [ ]:
market_returns-ff_data['RF']

In [ ]:
df_sentiment = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfTimeComplate.pkl")


In [ ]:
company="AAPL"

In [ ]:
df_dates= df_sentiment.loc[df_sentiment["Company"]==company]

In [ ]:
import pandas as pd

with open('/Users/Farid Srouji/Documents/Research/master/globaldata/companylist.txt') as f:
    companies = f.read().splitlines()
rows=[]
for company in companies:
    try:
        df_dates= df_sentiment.loc[df_sentiment["Company"]==company]#.groupby("Date").count()
        df_dates = df_dates.rename(columns={"Date":"date"})
        Stock = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/Data/PKL/Stocks/"+company+".pkl")

        Stock["date"] = pd.to_datetime(Stock["date"])
        df_Stock = pd.DataFrame(data=Stock[["date","adjClose"]]).rename(columns={0:"adjClose"})
        df_Stock["adjClose(t+1)"]=df_Stock["adjClose"].shift(periods=-1)
        df_Stock["Stock returns"]=np.log(df_Stock["adjClose(t+1)"]/df_Stock["adjClose"])
        df = df_Stock[["date","Stock returns"]]
        # df_dates["date"] = [date_obj .strftime('%Y-%m-%d') for date_obj  in df_dates.date]
        # df["date"] = [date_obj .strftime('%Y-%m-%d') for date_obj  in df.date]
        df["Stock NR"] = df["Stock returns"].mean()
        df["Stock AR"] = df["Stock returns"] - df["Stock NR"]

        Stock_stdev = 2*statistics.stdev(df["Stock AR"].dropna())
        Stock_mean = df["Stock AR"].mean()
        Stock = df.loc[(df["Stock AR"] >= Stock_mean+Stock_stdev) | (df["Stock AR"] <= Stock_mean-Stock_stdev)]
        mar_len = len(Stock.date)
        pr_len=len(df_dates.date)
        intr_len =len(set(df_dates.date).intersection(set(Stock.date)))
        one = [company,intr_len/mar_len,intr_len/pr_len,intr_len,pr_len,mar_len]
        rows.append(one)
    except:
        print("Company = "+company) 

In [ ]:
df_final = pd.DataFrame(data=rows).rename(columns={0:"Ticker",1:"intr to MAR",2:"intr to PR",3:"intr",4:"PR",5:"MAR"})

In [ ]:
df_final.mean()

In [ ]:
df_final["PR"].mean()